In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
data_news= pd.read_csv("C:/Users/WQM/Desktop/本科毕业论文/数据集/FNSPID/All_external.csv", dtype= str)
data_news= pd.DataFrame(data_news)

In [16]:
data_news_sim= data_news.loc[:, ["Date",  "Stock_symbol", "Article_title", "Article", "Url", "Publisher", "Author"]]

In [18]:
data_news_sim['Date'] = data_news_sim['Date'].str.split(' ').str[0]

In [20]:
# 1. 删除三列都为空值的行
df_cleaned = data_news_sim.dropna(subset=['Stock_symbol', 'Article_title', 'Article'], how='all')

# 2. 提取 Stock_symbol 有值 且 Article_title 或 Article 有值的行
data_stocknews = df_cleaned[df_cleaned['Stock_symbol'].notna() & (df_cleaned['Article_title'].notna() | df_cleaned['Article'].notna())]

# 3. 提取 Stock_symbol 为空 且 Article_title 或 Article 有值的行
data_purenews = df_cleaned[df_cleaned['Stock_symbol'].isna() & (df_cleaned['Article_title'].notna() | df_cleaned['Article'].notna())]

In [28]:
import re

# 定义一个函数来检查一行是否包含俄语字符
def contains_russian(text):
    # 如果文本为空或不是字符串，返回 False
    if pd.isna(text):
        return False
    # 使用正则表达式检查是否包含俄语字符
    return bool(re.search(r'[\u0400-\u04FF]', text))

# 过滤掉包含俄语字符的行
data_stocknews = data_stocknews[~data_stocknews['Article_title'].apply(contains_russian)]

In [ ]:
data_stocknews.drop_duplicates(inplace= True)
data_purenews.drop_duplicates(inplace= True)

In [33]:
data_stocknews.to_csv("Stocknews.csv", index=False)

In [34]:
data_purenews.to_csv("Purenews.csv", index=False)

In [2]:
# 定义文件夹路径
folder_path = "C:/Users/WQM/Desktop/本科毕业论文/数据集/FNSPID/full_history"

# 获取文件夹中的所有CSV文件名
files = [f for f in os.listdir(folder_path)]

# 创建一个空的列表来存储所有数据框
df_list = []

# 遍历每一个CSV文件
for file in files:
    # 获取文件路径
    file_path = os.path.join(folder_path, file)
    
    # 读取CSV文件
    df = pd.read_csv(file_path, dtype=str)
    
    # 获取文件名（去掉后缀名）
    file_name = os.path.splitext(file)[0]
    
    # 添加一列，列名为 'Stock_symbol'，并填充文件名到每一行
    df['Stock_symbol'] = file_name
    
    # 将当前的DataFrame添加到列表中
    df_list.append(df)

# 使用pandas的concat方法将所有DataFrame纵向合并（相同的列会自动对齐）
combined_df = pd.concat(df_list, ignore_index=True)

In [13]:
combined_df.to_csv("Stock_price.csv", index=False)